# Bias-Variance Tradeoff & Overfitting

This notebook provides a comprehensive exploration of one of the most fundamental concepts in machine learning: the **bias-variance tradeoff** and its practical manifestations as **overfitting** and **underfitting**.

**Learning Objectives:**
1. Understand intuitively what bias and variance mean
2. Visualize how model complexity affects both bias and variance
3. Identify underfitting (high bias) vs overfitting (high variance)
4. Recognize the tradeoff and find the "sweet spot"
5. Apply these concepts to regression and classification
6. Diagnose model performance issues and select appropriate solutions

## Why This Matters

The bias-variance tradeoff explains:
- Why simple models sometimes work better than complex ones
- Why more training data helps (and when it doesn't)
- How to choose the right model complexity
- Why regularization techniques work
- The fundamental limits of prediction accuracy

**Real-world analogy - Archery:**
- **High bias** = consistently missing the target in the same direction (systematic error)
- **High variance** = shots scattered all over (inconsistent, sensitive to small changes)
- **Good model** = tight grouping near the bullseye (low bias, low variance)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.datasets import make_moons
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")

---
# Part 1: The Core Problem - Underfitting vs Overfitting

When training ML models, we face a delicate balance:

| Problem | Cause | Symptom |
|---------|-------|--------|
| **Underfitting** | Model too simple | High error on both train & test |
| **Overfitting** | Model too complex | Low train error, high test error |
| **Just Right** | Balanced complexity | Low error on both |

In [ ]:
# Generate synthetic data with known underlying pattern
def true_function(x):
    """The underlying true function: cubic polynomial"""
    return 0.5 + 1.5 * x - 3 * x**2 + 2 * x**3

def generate_data(n_samples=100, noise=0.1):
    """Generate data from true function with noise."""
    X = np.linspace(0, 1, n_samples)
    y_true = true_function(X)
    y = y_true + np.random.normal(0, noise, n_samples)
    return X.reshape(-1, 1), y, y_true

# Generate train and test sets
X_train, y_train, y_train_true = generate_data(n_samples=30, noise=0.1)
X_test, y_test, y_test_true = generate_data(n_samples=100, noise=0.1)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Visualize the data
plt.figure(figsize=(10, 5))
plt.scatter(X_train, y_train, alpha=0.6, s=50, label='Training data', color='blue')
plt.plot(X_test, y_test_true, 'g--', linewidth=2, label='True function', alpha=0.8)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Training Data and True Underlying Function')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Polynomial Regression: A Model Capacity Playground

Polynomial degree controls **model capacity**:
- **Degree 1** (Linear): Very limited capacity → prone to underfitting
- **Degree 3** (Cubic): Matches true function → good fit
- **Degree 15+**: Excessive capacity → prone to overfitting

In [ ]:
def fit_polynomial(X_train, y_train, degree):
    """Fit a polynomial regression model."""
    poly_features = PolynomialFeatures(degree=degree, include_bias=True)
    X_poly = poly_features.fit_transform(X_train)
    model = LinearRegression()
    model.fit(X_poly, y_train)
    return model, poly_features

def predict_polynomial(model, poly_features, X):
    """Make predictions with polynomial model."""
    X_poly = poly_features.transform(X)
    return model.predict(X_poly)

In [ ]:
# Fit models of different complexity
degrees = [1, 3, 15]
models = {}

for degree in degrees:
    model, poly_feat = fit_polynomial(X_train, y_train, degree)
    y_train_pred = predict_polynomial(model, poly_feat, X_train)
    y_test_pred = predict_polynomial(model, poly_feat, X_test)
    
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    
    models[degree] = {
        'model': model,
        'poly_feat': poly_feat,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'predictions': y_test_pred
    }
    
    print(f"Degree {degree:2d}: Train MSE = {train_mse:.4f}, Test MSE = {test_mse:.4f}")

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
labels = ['UNDERFITTING\n(High Bias)', 'GOOD FIT\n(Balanced)', 'OVERFITTING\n(High Variance)']

for ax, degree, label in zip(axes, degrees, labels):
    m = models[degree]
    ax.scatter(X_train, y_train, alpha=0.6, s=50, color='blue', zorder=3)
    ax.plot(X_test, y_test_true, 'g--', linewidth=2, alpha=0.6, label='True function')
    ax.plot(X_test, m['predictions'], 'r-', linewidth=2, label=f'Degree {degree}')
    ax.set_xlabel('X')
    ax.set_ylabel('y')
    ax.set_title(f'{label}\nTrain MSE: {m["train_mse"]:.4f}, Test MSE: {m["test_mse"]:.4f}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    if degree == 15:
        ax.set_ylim(-1, 2)

plt.tight_layout()
plt.show()

**Key Observations:**
- **Underfitting (Degree 1)**: Both train and test error high. Model can't capture the pattern.
- **Good Fit (Degree 3)**: Low train and test error. Model captures the true pattern.
- **Overfitting (Degree 15)**: Very low train error, higher test error. Model memorizes noise.

---
# Part 2: The Bias-Variance Decomposition

The expected prediction error can be decomposed:

$$\text{Expected Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$$

| Term | Meaning | Cause |
|------|---------|-------|
| **Bias²** | Systematic error from wrong assumptions | Model too simple |
| **Variance** | Sensitivity to training data fluctuations | Model too complex |
| **Irreducible** | Inherent noise in data | Can't be reduced |

In [ ]:
# Visualize the conceptual tradeoff
model_complexity = np.linspace(0, 10, 100)

bias_squared = 5 * np.exp(-model_complexity/2) + 0.1
variance = 0.05 * model_complexity**2
total_error = bias_squared + variance
irreducible = np.ones_like(model_complexity) * 0.1

optimal_idx = np.argmin(total_error)

fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(model_complexity, bias_squared, 'r-', linewidth=3, label='Bias²', alpha=0.8)
ax.plot(model_complexity, variance, 'b-', linewidth=3, label='Variance', alpha=0.8)
ax.plot(model_complexity, total_error, 'purple', linewidth=4, label='Total Error', alpha=0.9)
ax.plot(model_complexity, irreducible, 'g--', linewidth=2, label='Irreducible Error', alpha=0.7)

ax.axvline(model_complexity[optimal_idx], color='black', linestyle=':', linewidth=2)
ax.scatter([model_complexity[optimal_idx]], [total_error[optimal_idx]], 
           color='purple', s=200, marker='*', edgecolors='black', linewidth=2, zorder=10)

ax.axvspan(0, 3, alpha=0.1, color='red')
ax.axvspan(7, 10, alpha=0.1, color='blue')

ax.set_xlabel('Model Complexity', fontsize=13, fontweight='bold')
ax.set_ylabel('Error', fontsize=13, fontweight='bold')
ax.set_title('The Bias-Variance Tradeoff', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 10])
ax.set_ylim([0, 6])

ax.text(1.5, 5, 'HIGH BIAS\n(Underfitting)', ha='center', fontsize=11, color='darkred', fontweight='bold')
ax.text(8.5, 5, 'HIGH VARIANCE\n(Overfitting)', ha='center', fontsize=11, color='darkblue', fontweight='bold')

plt.tight_layout()
plt.show()

---
# Part 3: Empirical Bias-Variance Measurement

Let's **measure** bias and variance by training multiple models on different samples.

In [ ]:
# Generate multiple training sets and measure bias/variance
n_datasets = 50
n_train = 30
n_test = 200

X_test_fixed = np.linspace(0, 1, n_test).reshape(-1, 1)
y_test_fixed = true_function(X_test_fixed.ravel())

degrees_to_test = [1, 2, 3, 5, 7, 9, 12, 15]
predictions_by_degree = {}

for degree in degrees_to_test:
    predictions_list = []
    poly_features = PolynomialFeatures(degree=degree, include_bias=True)
    X_test_poly = poly_features.fit_transform(X_test_fixed)
    
    for i in range(n_datasets):
        X_train_i = np.linspace(0, 1, n_train).reshape(-1, 1)
        y_train_i = true_function(X_train_i.ravel()) + np.random.normal(0, 0.1, n_train)
        
        X_train_poly = poly_features.fit_transform(X_train_i)
        model = LinearRegression()
        model.fit(X_train_poly, y_train_i)
        
        y_pred = model.predict(X_test_poly)
        predictions_list.append(y_pred)
    
    predictions_by_degree[degree] = np.array(predictions_list)

print(f"Generated {n_datasets} models for each of {len(degrees_to_test)} polynomial degrees")

In [ ]:
# Calculate empirical bias and variance
bias_values = []
variance_values = []
total_error_values = []

print("Empirical Bias-Variance Decomposition:\n")
for degree in degrees_to_test:
    predictions = predictions_by_degree[degree]
    mean_prediction = np.mean(predictions, axis=0)
    
    bias_squared = np.mean((mean_prediction - y_test_fixed)**2)
    variance = np.mean(np.var(predictions, axis=0))
    total_error = bias_squared + variance
    
    bias_values.append(bias_squared)
    variance_values.append(variance)
    total_error_values.append(total_error)
    
    print(f"Degree {degree:2d}: Bias² = {bias_squared:.4f}, Variance = {variance:.4f}, Total = {total_error:.4f}")

In [ ]:
# Plot empirical bias-variance tradeoff
optimal_idx = np.argmin(total_error_values)
optimal_degree = degrees_to_test[optimal_idx]

fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(degrees_to_test, bias_values, 'r-o', linewidth=3, markersize=8, label='Bias²', alpha=0.8)
ax.plot(degrees_to_test, variance_values, 'b-s', linewidth=3, markersize=8, label='Variance', alpha=0.8)
ax.plot(degrees_to_test, total_error_values, 'purple', marker='D', linewidth=4, markersize=10, 
        label='Total Error', alpha=0.9)

ax.axvline(optimal_degree, color='black', linestyle=':', linewidth=2)
ax.scatter([optimal_degree], [total_error_values[optimal_idx]], color='purple', s=300, 
           marker='*', edgecolors='black', linewidth=2, zorder=10)

ax.set_xlabel('Polynomial Degree (Model Complexity)', fontsize=13, fontweight='bold')
ax.set_ylabel('Error', fontsize=13, fontweight='bold')
ax.set_title('Empirical Bias-Variance Tradeoff', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xticks(degrees_to_test)

plt.tight_layout()
plt.show()

print(f"\nOptimal polynomial degree: {optimal_degree}")

---
# Part 4: Training vs Validation Error Curves

The classic diagnostic tool: plot training and validation error vs model complexity.

In [ ]:
# Calculate training and validation errors
degrees = range(1, 21)
train_errors = []
test_errors = []

for degree in degrees:
    model, poly_feat = fit_polynomial(X_train, y_train, degree)
    
    y_train_pred = predict_polynomial(model, poly_feat, X_train)
    y_test_pred = predict_polynomial(model, poly_feat, X_test)
    
    train_errors.append(mean_squared_error(y_train, y_train_pred))
    test_errors.append(mean_squared_error(y_test, y_test_pred))

In [ ]:
# Plot training vs test error
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(list(degrees), train_errors, 'b-o', linewidth=3, markersize=6, label='Training Error', alpha=0.8)
ax.plot(list(degrees), test_errors, 'r-s', linewidth=3, markersize=6, label='Test Error', alpha=0.8)

optimal_idx = np.argmin(test_errors)
optimal_degree = list(degrees)[optimal_idx]
ax.axvline(optimal_degree, color='green', linestyle='--', linewidth=2, alpha=0.7)
ax.scatter([optimal_degree], [test_errors[optimal_idx]], color='green', s=200, 
           marker='*', edgecolors='black', linewidth=2, zorder=10, label=f'Optimal (degree {optimal_degree})')

ax.axvspan(1, 3, alpha=0.1, color='yellow')
ax.axvspan(10, 20, alpha=0.1, color='orange')

ax.set_xlabel('Polynomial Degree (Model Complexity)', fontsize=13, fontweight='bold')
ax.set_ylabel('Mean Squared Error', fontsize=13, fontweight='bold')
ax.set_title('Training vs Test Error', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

ax.text(2, max(test_errors)*0.3, 'UNDERFITTING', fontsize=10, ha='center', color='darkred', fontweight='bold')
ax.text(15, max(test_errors)*0.3, 'OVERFITTING', fontsize=10, ha='center', color='darkred', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nKey Patterns:")
print("1. Training error always decreases with complexity")
print("2. Test error has U-shape: decreases then increases")
print("3. Gap between curves indicates overfitting")
print(f"4. Optimal degree: {optimal_degree}")

---
# Part 5: Learning Curves - Effect of Training Data Size

**Key insight**: More training data helps reduce **variance** (overfitting) but NOT **bias** (underfitting)!

In [ ]:
def compute_learning_curve(degree, sample_sizes):
    """Compute learning curve for given polynomial degree."""
    train_errors = []
    val_errors = []
    
    X_val = np.linspace(0, 1, 100).reshape(-1, 1)
    y_val = true_function(X_val.ravel())
    
    for n in sample_sizes:
        X_train = np.linspace(0, 1, n).reshape(-1, 1)
        y_train = true_function(X_train.ravel()) + np.random.normal(0, 0.1, n)
        
        model, poly_feat = fit_polynomial(X_train, y_train, degree)
        
        y_train_pred = predict_polynomial(model, poly_feat, X_train)
        y_val_pred = predict_polynomial(model, poly_feat, X_val)
        
        train_errors.append(mean_squared_error(y_train, y_train_pred))
        val_errors.append(mean_squared_error(y_val, y_val_pred))
    
    return train_errors, val_errors

sample_sizes = [10, 20, 30, 50, 75, 100, 150, 200]

In [ ]:
# Plot learning curves for different complexities
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
degrees_lc = [1, 5, 15]
titles = ['Degree 1 (HIGH BIAS)', 'Degree 5 (GOOD)', 'Degree 15 (HIGH VARIANCE)']

for ax, degree, title in zip(axes, degrees_lc, titles):
    train_err, val_err = compute_learning_curve(degree, sample_sizes)
    
    ax.plot(sample_sizes, train_err, 'b-o', linewidth=2, markersize=6, label='Training Error')
    ax.plot(sample_sizes, val_err, 'r-s', linewidth=2, markersize=6, label='Validation Error')
    
    ax.set_xlabel('Training Set Size', fontsize=12)
    ax.set_ylabel('Mean Squared Error', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 0.3])

plt.tight_layout()
plt.show()

print("\nLearning Curve Interpretation:")
print("• HIGH BIAS (degree 1): Both errors high and flat. More data DOES NOT help!")
print("• GOOD MODEL (degree 5): Errors converge to low values. More data helps.")
print("• HIGH VARIANCE (degree 15): Large gap. More data helps reduce the gap.")

---
# Part 6: k-Nearest Neighbors Classification Example

The bias-variance tradeoff applies to **all** ML algorithms. In k-NN:
- **Small k** (e.g., k=1): High complexity → LOW BIAS, HIGH VARIANCE
- **Large k** (e.g., k=100): Low complexity → HIGH BIAS, LOW VARIANCE

In [ ]:
# Generate classification dataset
X_class, y_class = make_moons(n_samples=200, noise=0.25, random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_class, y_class, test_size=0.3, random_state=42)

def plot_decision_boundary(X, y, model, ax, title):
    """Plot decision boundary for 2D classification."""
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=30, alpha=0.7, edgecolors='black', linewidth=0.5)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

In [ ]:
# Compare different k values
k_values = [1, 5, 20, 50]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for ax, k in zip(axes, k_values):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_c, y_train_c)
    
    train_acc = knn.score(X_train_c, y_train_c)
    test_acc = knn.score(X_test_c, y_test_c)
    
    regime = "HIGH VARIANCE" if k == 1 else ("HIGH BIAS" if k == 50 else "Good")
    title = f'k = {k} ({regime})\nTrain: {train_acc:.3f}, Test: {test_acc:.3f}'
    plot_decision_boundary(X_train_c, y_train_c, knn, ax, title)

plt.tight_layout()
plt.show()

print("\nObservations:")
print("• k=1: Very jagged boundary (HIGH VARIANCE)")
print("• k=5-20: Smooth but flexible boundary (GOOD)")
print("• k=50: Very smooth, misses patterns (HIGH BIAS)")

---
# Part 7: Practical Diagnosis Guide

## Decision Tree

```
Is training error high?
│
├─ YES: Is test error also high?
│   ├─ YES → HIGH BIAS (underfitting)
│   │        Solutions: more complex model, more features, less regularization
│   └─ NO → Unusual. Check data quality.
│
└─ NO: Is there a large gap to test error?
    ├─ YES → HIGH VARIANCE (overfitting)
    │        Solutions: more data, simpler model, regularization
    └─ NO → Good fit! Model is well-tuned.
```

## Solutions Summary

### If Underfitting (High Bias):
- ✅ Increase model complexity
- ✅ Add more features
- ✅ Reduce regularization
- ✅ Train longer
- ❌ More data does NOT help!

### If Overfitting (High Variance):
- ✅ Get more training data (most effective!)
- ✅ Decrease model complexity
- ✅ Add regularization (L1, L2, dropout)
- ✅ Feature selection
- ✅ Early stopping
- ✅ Ensemble methods

---
# Part 8: Effect of Regularization

Regularization provides fine-grained control over the bias-variance tradeoff.

In [ ]:
# Effect of L2 regularization on high-degree polynomial
degree = 15
alphas = [0, 0.001, 0.01, 0.1, 1.0, 10.0]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

poly_features = PolynomialFeatures(degree=degree, include_bias=True)
X_train_poly = poly_features.fit_transform(X_train)
X_test_poly = poly_features.transform(X_test)

for ax, alpha in zip(axes, alphas):
    if alpha == 0:
        model = LinearRegression()
    else:
        model = Ridge(alpha=alpha)
    
    model.fit(X_train_poly, y_train)
    y_pred = model.predict(X_test_poly)
    
    train_mse = mean_squared_error(y_train, model.predict(X_train_poly))
    test_mse = mean_squared_error(y_test, y_pred)
    
    ax.scatter(X_train, y_train, color='blue', s=40, alpha=0.6, label='Training data')
    ax.plot(X_test, y_test_true, 'g--', linewidth=2, label='True function', alpha=0.7)
    ax.plot(X_test, y_pred, 'r-', linewidth=2, label='Model', alpha=0.9)
    ax.set_xlabel('X')
    ax.set_ylabel('y')
    ax.set_title(f'α = {alpha}\nTrain: {train_mse:.4f}, Test: {test_mse:.4f}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([-2, 2])

plt.tight_layout()
plt.show()

print("\nRegularization Effect:")
print("• α = 0: No regularization → Wild oscillations (overfitting)")
print("• Small α: Mild regularization → Smoother, better generalization")
print("• Large α: Strong regularization → Very smooth, may underfit")

---
# Summary: Key Takeaways

## The Fundamental Tradeoff

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$$

- **You can't minimize both bias and variance simultaneously**
- **The goal is to find the sweet spot** that minimizes total error

## Quick Reference

| Symptom | Diagnosis | Solution |
|---------|-----------|----------|
| High train error, High test error | HIGH BIAS | More complexity, more features |
| Low train error, High test error | HIGH VARIANCE | More data, regularization, simpler model |
| Low train error, Low test error | GOOD FIT | You're done! |

## Connection to Other Concepts

- **Regularization**: Increases bias to reduce variance
- **Ensemble Methods**: Bagging reduces variance, Boosting reduces bias
- **Cross-Validation**: Estimates generalization error to find optimal complexity
- **Deep Learning**: High capacity (low bias potential), requires regularization for variance

**Remember**: The goal isn't to minimize training error — it's to build models that **generalize** well to unseen data!